In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch

torch.cuda.is_available()

True

In [38]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from enum import Enum
from dataclasses import dataclass, field
from collections import defaultdict
from typing import Optional, Union, Any, Callable
import pandas as pd

In [42]:
df = pd.DataFrame({
    'item': [[0,1,2], [1,2], [1,1,1,2]],
    'age': [[25,26,27], [35,37], [41,41,42,55]],
    'price': [[100, 110, 120], [1000, 1500], [1000, 1000, 1000, 2500]],
})
df

,item,age,price
0,"[0, 1, 2]","[25, 26, 27]","[100, 110, 120]"
1,"[1, 2]","[35, 37]","[1000, 1500]"
2,"[1, 1, 1, 2]","[41, 41, 42, 55]","[1000, 1000, 1000, 2500]"


In [50]:
df.to_dict(orient='list')

{'item': [[0, 1, 2], [1, 2], [1, 1, 1, 2]],
 'age': [[25, 26, 27], [35, 37], [41, 41, 42, 55]],
 'price': [[100, 110, 120], [1000, 1500], [1000, 1000, 1000, 2500]]}

In [56]:
len(df.values.tolist())

3

In [18]:
class BaseDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

In [117]:
COLLATE_REGISTRY = {}

def register_collate(name:str=None):
    def decorator(func:Callable):
        key = name or func.__name__
        if key not in COLLATE_REGISTRY:
            COLLATE_REGISTRY[name] = func
        return func
    return decorator

class CollateType(Enum):
    COMMON = 'common'
    DELTA = 'delta'
    DELTA_BIN = 'delta_bin'

class FeatureDtype(Enum):
    LONG = 'long'
    FLOAT = 'float'

DTYPE = {
    FeatureDtype.LONG:torch.long,
    FeatureDtype.FLOAT:torch.float,
}

@dataclass
class Feature:
    name:str
    d_model:int
    max_len:int
    x_dtype:FeatureDtype
    y_dtype:FeatureDtype
    is_linear:bool = field(default=False)
    collate:CollateType = field(default=CollateType.COMMON)
    vocab_size:Optional[int] = field(default=None)

    def __post_init__(self):
        self.x_dtype = DTYPE[self.x_dtype]
        self.y_dtype = DTYPE[self.y_dtype]

def _pack_xy(b, dtype):
    return torch.tensor(b, dtype=dtype)

def collate_common(batch, x_dtype, y_dtype, inference=False):
    if inference:
        return [_pack_x(b, x_dtype) for b in batch], None
    
    x = [_pack_xy(b[:-1], x_dtype) for b in batch]
    y = [_pack_xy(b[-1], y_dtype) for b in batch]
    return x, y

def collate_delta(batch, x_dtype, y_dtype, inference=False):
    if inference:
        return [_pack_x(b, x_dtype) for b in batch], None

    x = [_pack_xy(b[:-1], x_dtype) for b in batch]
    y = [_pack_xy(b[-1]-b[-2], y_dtype) for b in batch]
    return x, y

def collate_delta_bin(batch, x_dtype, y_dtype, inference=False):
    if inference:
        return [_pack_x(b, x_dtype) for b in batch], None

    def logic(b):
        if b <= 1:
            return 0
        if b <= 3:
            return 1
        return 2

    x = [_pack_xy(b[:-1], x_dtype) for b in batch]
    y = [_pack_xy(logic(b[-1]-b[-2]), y_dtype) for b in batch]
    return x, y

def auto_collate(batch, features, inference=False):
    X = defaultdict(list)
    y = defaultdict(list)
    for feature in features:
        _x, _y = COLLATE_REGISTRY[feature.collate](batch=[b[feature.name] for b in batch], x_dtype=feature.x_dtype, y_dtype=feature.y_dtype, inference=inference)
        X[feature.name].extend(_x)
        y[feature.name].extend(_y)
    for feature in features:
        _x = pad_sequence(X[feature.name], batch_first=True)
        _y = torch.tensor(y[feature.name])
        # how can I know which one will be linear or embedding?
        if feature.is_linear:
            _x = _x.unsqueeze(-1)
        X[feature.name] = _x
        y[feature.name] = _y
    return dict(X=X, y=y, mask=None)

register_collate(CollateType.COMMON)(collate_common)
register_collate(CollateType.DELTA)(collate_delta)
register_collate(CollateType.DELTA_BIN)(collate_delta_bin)

COLLATE_REGISTRY

{<CollateType.COMMON: 'common'>: <function __main__.collate_common(batch, x_dtype, y_dtype, inference=False)>,
 <CollateType.DELTA: 'delta'>: <function __main__.collate_delta(batch, x_dtype, y_dtype, inference=False)>,
 <CollateType.DELTA_BIN: 'delta_bin'>: <function __main__.collate_delta_bin(batch, x_dtype, y_dtype, inference=False)>}

In [118]:
features = [
    Feature(name='item', d_model=10, max_len=10, vocab_size=10, x_dtype=FeatureDtype.LONG, y_dtype=FeatureDtype.LONG, collate=CollateType.COMMON),
    Feature(name='age', d_model=10, max_len=10, vocab_size=10, x_dtype=FeatureDtype.LONG, y_dtype=FeatureDtype.LONG, collate=CollateType.DELTA_BIN, is_linear=True),
    Feature(name='price', d_model=10, max_len=10, vocab_size=10, x_dtype=FeatureDtype.FLOAT, y_dtype=FeatureDtype.FLOAT, collate=CollateType.DELTA, is_linear=True),
]

In [119]:
df.to_dict(orient='records')

[{'item': [0, 1, 2], 'age': [25, 26, 27], 'price': [100, 110, 120]},
 {'item': [1, 2], 'age': [35, 37], 'price': [1000, 1500]},
 {'item': [1, 1, 1, 2],
  'age': [41, 41, 42, 55],
  'price': [1000, 1000, 1000, 2500]}]

In [120]:
data = df.to_dict(orient='records')

dataset = BaseDataset(data)

batch = dataset[:]

In [123]:
prep_data = auto_collate(batch, features, inference=False)

In [127]:
prep_data['X']['item'].shape

torch.Size([3, 3])